# pragmatic-dimensions-classifier — Phase 1: Data Familiarization & Preliminary Baselines

**Status:** Exploratory. See `TASK.md` for the full brief.

This notebook loads two open formality datasets — Pavlick & Tetreault (2016) and SQUINKY! (Lahiri 2015) — inspects their structure, and runs a few preliminary classifier tests. The goal is to get hands-on with both corpora and surface concrete signal (especially: how hard is cross-corpus generalization?) before finalizing the design of the actual formality/register classifier project.

**Not the final design.** Binarization thresholds, model choice, and preprocessing here are deliberately simple — good enough to see signal, not good enough to ship.

## 0. Setup

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
DATA_DIR = "../data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Load Pavlick & Tetreault (2016)

License: CC-BY-3.0. Columns: `domain` (news / blog / email / answers), `avg_score` (float, −3 to 3, lower = less formal), `sentence`. See `data/README.md` for citation requirements (cite both Pavlick & Tetreault 2016 and Lahiri 2015).

In [ ]:
pt_dataset = load_dataset("osyvokon/pavlick-formality-scores")
pt_train = pt_dataset["train"].to_pandas()
pt_test = pt_dataset["test"].to_pandas()
pt_all = pd.concat([pt_train, pt_test], ignore_index=True)

print(pt_all.shape)
print(pt_all["domain"].value_counts())
pt_all.head()

### 1.1 Inspect distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(pt_all["avg_score"], bins=30, ax=axes[0])
axes[0].set_title("Pavlick-Tetreault: formality score distribution (all genres)")

sns.boxplot(data=pt_all, x="domain", y="avg_score", ax=axes[1])
axes[1].set_title("Formality by genre")
plt.tight_layout()
plt.show()

pt_all.groupby("domain")["avg_score"].describe()

✍️ **Your observations:** _(genre patterns, anything surprising, anything that affects later design)_

## 2. Load SQUINKY! (Lahiri 2015)

**Column structure is not fully documented** — see `TASK.md` for what's known from the reference implementation (`meyersbs/squinky`). Inspect before assuming anything. The raw file is hosted on a personal academic homepage, not a stable archive — if the download below fails, check the GitHub repo's issues for a mirror.

In [ ]:
SQUINKY_URL = "http://people.rc.rit.edu/~bsm9339/corpora/squinky_corpus/mturk_merged.csv"
SQUINKY_PATH = os.path.join(DATA_DIR, "squinky_mturk_merged.csv")

if not os.path.exists(SQUINKY_PATH):
    resp = requests.get(SQUINKY_URL, timeout=30)
    resp.raise_for_status()
    with open(SQUINKY_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded to {SQUINKY_PATH}")
else:
    print("Already downloaded.")

### 2.1 Inspect raw structure before assuming anything

In [ ]:
squinky_raw = pd.read_csv(SQUINKY_PATH)
print(squinky_raw.shape)
print(squinky_raw.columns.tolist())
squinky_raw.head()

**Note:** if the printed columns above aren't self-explanatory, rename them based on position, following the convention used in the reference implementation (`meyersbs/squinky/squinky/classifier.py`): column 0 = id, column 1 = formality (1–7), column 2 = informativeness (1–7), column 3 = implicature (1–7), last column = sentence text. There may be additional columns (e.g. genre/source) in between that aren't documented anywhere — confirm against the printed output above, don't assume.

In [ ]:
# TODO: confirm against the printed structure above, then adjust if needed.
squinky = squinky_raw.copy()

# Example rename — VERIFY before trusting:
# squinky = squinky.rename(columns={
#     squinky.columns[0]: "id",
#     squinky.columns[1]: "formality",
#     squinky.columns[2]: "informativeness",
#     squinky.columns[3]: "implicature",
#     squinky.columns[-1]: "sentence",
# })
squinky.head()

### 2.2 Inspect distributions (formality, informativeness, implicature)

In [ ]:
# TODO: once columns are confirmed/renamed, mirror the Pavlick-Tetreault distribution
# plots above for all three SQUINKY! dimensions (formality, informativeness, implicature).
# Pay particular attention to genre-wise variation if a genre/source column exists,
# and to the known reliability gap: Lahiri (2015) reports implicature ratings are
# noticeably less reliable across annotator rounds than formality or informativeness.

✍️ **Your observations:** _(how does the 1–7 scale compare to PT's −3..3? any class imbalance? does implicature look noisier, as the paper warns?)_

## 3. Side-by-side comparison

In [ ]:
comparison = pd.DataFrame({
    "Pavlick-Tetreault (2016)": {
        "N sentences": len(pt_all),
        "Dimensions": "Formality",
        "Scale": "-3 to 3 (continuous)",
        "Genres": ", ".join(sorted(pt_all["domain"].unique())),
        "License": "CC-BY-3.0",
    },
    "SQUINKY! (Lahiri 2015)": {
        "N sentences": len(squinky),
        "Dimensions": "Formality, Informativeness, Implicature",
        "Scale": "1 to 7 (continuous)",
        "Genres": "news, blog, forum (per paper — confirm against data)",
        "License": "Code: MIT; data license unconfirmed",
    },
})
comparison

## 4. Preliminary classifier tests

Three quick tests, in increasing order of interest:

- **Model A** — TF-IDF + Logistic Regression, in-domain on Pavlick-Tetreault.
- **Model B** — TF-IDF + Logistic Regression, in-domain on SQUINKY! formality.
- **Model C** — cross-corpus: train on one, evaluate on the other. This is the diagnostic that actually matters for the final design — it tells us how viable "train on one corpus, evaluate cross-corpus on the other" is as a project framing, before we commit to it.

Binarization below is a simple median/threshold split, chosen for speed — **not** the final design decision. That's flagged explicitly at each step.

In [ ]:
def run_tfidf_logreg(train_texts, train_labels, test_texts, test_labels, label="Model"):
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, train_labels)
    preds = clf.predict(X_test)

    print(f"--- {label} ---")
    print(classification_report(test_labels, preds))
    print("Confusion matrix:")
    print(confusion_matrix(test_labels, preds))
    return clf, vectorizer, preds

### Model A — Pavlick-Tetreault, in-domain

In [ ]:
# Binarization choice (preliminary only — revisit in the design discussion):
# median split, since avg_score has no natural zero-formality cutoff across genres.
pt_threshold = pt_all["avg_score"].median()
pt_all["formal_label"] = (pt_all["avg_score"] > pt_threshold).astype(int)
print(f"PT median threshold: {pt_threshold}")
print(pt_all["formal_label"].value_counts())

pt_train_texts, pt_test_texts, pt_train_labels, pt_test_labels = train_test_split(
    pt_all["sentence"], pt_all["formal_label"],
    test_size=0.2, random_state=42, stratify=pt_all["formal_label"]
)

model_a, vec_a, preds_a = run_tfidf_logreg(
    pt_train_texts, pt_train_labels, pt_test_texts, pt_test_labels,
    label="Model A: Pavlick-Tetreault in-domain"
)

### Model B — SQUINKY!, in-domain

In [ ]:
# TODO: mirror Model A once SQUINKY! columns are confirmed/renamed above.
# Suggested binarization to start with (matches the reference implementation's
# convention, NOT necessarily what we'll use in the final project):
#
# squinky["formal_label"] = (squinky["formality"] > 4).astype(int)
#
# sq_train_texts, sq_test_texts, sq_train_labels, sq_test_labels = train_test_split(
#     squinky["sentence"], squinky["formal_label"],
#     test_size=0.2, random_state=42, stratify=squinky["formal_label"]
# )
#
# model_b, vec_b, preds_b = run_tfidf_logreg(
#     sq_train_texts, sq_train_labels, sq_test_texts, sq_test_labels,
#     label="Model B: SQUINKY! in-domain"
# )

### Model C — Cross-corpus generalization (the interesting one)

Train on one corpus, evaluate on the other. A large F1 drop relative to the in-domain models is expected — the question is *how* large, and whether it looks genre-driven, scale-driven, or both. This is the central diagnostic for whether "train on one, evaluate cross-corpus on the other" is viable as the eventual project framing, or whether it needs more careful harmonization first (rescaling, genre-matched subsets, vocabulary normalization).

In [ ]:
# TODO: once Model B's labels exist, run both directions:
#
# Direction 1 — train on PT, evaluate on SQUINKY!:
# vec_c1 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
# X_train_pt = vec_c1.fit_transform(pt_all["sentence"])
# clf_c1 = LogisticRegression(max_iter=1000).fit(X_train_pt, pt_all["formal_label"])
# X_test_squinky = vec_c1.transform(squinky["sentence"])
# preds_c1 = clf_c1.predict(X_test_squinky)
# print("--- Model C1: train PT -> test SQUINKY! ---")
# print(classification_report(squinky["formal_label"], preds_c1))
#
# Direction 2 — train on SQUINKY!, evaluate on PT (repeat symmetrically):
# vec_c2 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
# X_train_sq = vec_c2.fit_transform(squinky["sentence"])
# clf_c2 = LogisticRegression(max_iter=1000).fit(X_train_sq, squinky["formal_label"])
# X_test_pt = vec_c2.transform(pt_all["sentence"])
# preds_c2 = clf_c2.predict(X_test_pt)
# print("--- Model C2: train SQUINKY! -> test PT ---")
# print(classification_report(pt_all["formal_label"], preds_c2))
#
# Compare both directions' F1 against the in-domain Model A/B scores above.

✍️ **Your observations:** _(how much does cross-corpus accuracy drop? genre-driven or scale-driven? does this support the cross-corpus framing, or suggest a different angle?)_

## 5. Open questions for the design discussion

- **Classification vs. regression** — does the preliminary binarization above lose information that matters (e.g. near-threshold sentences)?
- **Binarization strategy** — median split, fixed threshold, or keep continuous throughout?
- **Cross-corpus viability** — how much does performance drop in Model C, and is it worth addressing before treating "cross-corpus generalization" as the headline framing of the project?
- **Implicature reliability** — Lahiri (2015) reports implicature ratings are less reliable than formality/informativeness. Exclude it, report it with a caveat, or treat it as a qualitative bonus analysis rather than a classifier target?
- **Data licensing** — confirm SQUINKY! data licensing before any public release of code or results tied to it (see `data/README.md`).

## 6. Next steps

Bring the observations and open questions above into the design discussion. From there: lock in the binarization/regression approach, decide the final cross-corpus architecture (if pursued), and move on to the transformer fine-tuning stage (fast.ai Lesson 4 as the primary reference).